In [2]:
import yfinance as yf
import pandas as pd
import time
import requests

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})

tickers = ["AAPL", "MSFT", "GOOGL", "JPM", "XOM",'SPY']
data = {}

for ticker in tickers:
    data[ticker] = yf.download(ticker, start="2020-01-01", end="2024-12-31", 
                                auto_adjust=True, progress=False, session=session)
    time.sleep(2)  # 每只间隔2秒

# 合并收盘价
close = pd.DataFrame({t: data[t]["Close"].squeeze() for t in tickers})
print(close.shape)
close.head()

(1257, 6)


,AAPL,MSFT,GOOGL,JPM,XOM,SPY
Date,,,,,,
2020-01-02,72.333862,151.829514,67.873032,118.430321,52.947029,296.888184
2020-01-03,71.630623,149.939026,67.517952,116.867455,52.521362,294.640045
2020-01-06,72.201408,150.326553,69.317589,116.774559,52.924633,295.764160
2020-01-07,71.861855,148.955917,69.183701,114.789314,52.491493,294.932556
2020-01-08,73.017830,151.328537,69.676117,115.684761,51.699913,296.504395


In [1]:
import sys
print(sys.executable)

E:\Handout\project\project_quant\quant-factor-mining\.venv\Scripts\python.exe


- 计算日收益率

In [15]:
daily_return=pd.DataFrame()
for ticker in tickers:
    daily_return[f'{ticker}']=close[ticker].pct_change()

In [16]:
print(daily_return.head())

                AAPL      MSFT     GOOGL       JPM       XOM       SPY
Date                                                                  
2020-01-02       NaN       NaN       NaN       NaN       NaN       NaN
2020-01-03 -0.009722 -0.012451 -0.005232 -0.013197 -0.008039 -0.007572
2020-01-06  0.007968  0.002585  0.026654 -0.000795  0.007678  0.003815
2020-01-07 -0.004703 -0.009118 -0.001932 -0.017001 -0.008184 -0.002812
2020-01-08  0.016086  0.015928  0.007118  0.007801 -0.015080  0.005329


- 动量因子=(第一天-最后一天)/第一天

In [ ]:
def momentum(df,ticker, day):
    if ticker in tickers:
        current_price=df[ticker]
        start_price=df[ticker].shift(day-1)
        df[f"momentum_{ticker}"]=(start_price-current_price)/start_price
        return df

- 反转因子:
- 短期反转因子捕捉股票价格在短期内（通常是一个月内）的表现，并预期这种表现会在短期内发生反转。简而言之，近期表现较差的股票预计在未来短期内表现较好，而近期表现较好的股票预计在未来短期内表现较差。短期反转因子作为动量因子家族的新成员，是对Barra风险模型的补充，可以有效地分散风险并提高整体策略的稳健性。
- 1、数据准备：收集过去一个月内每天的股票收盘价和市场指数收盘价。

- 2、计算超额收益：对于每只股票，计算其过去一个月相对于市场指数的超额收益（Excess Return）。


- 3、指数加权和：计算超额收益的指数加权和（Exponentially Weighted Sum），使用63个交易日的窗口期，半衰期为10天。

其中， 

- 4、反转符号：将上述结果乘以-1来反转符号，使正的值表示预期的正向反转，负的值表示预期的负向反转。
- 5、移动平均：计算过去三天的等权重移动平均值，滞后一天。

In [14]:
market_return=daily_return['SPY_daily_return']
excess_return=daily_return.drop(columns=['SPY_daily_return']).sub(market_return, axis=0)
print(excess_return.head())

            AAPL_daily_return  MSFT_daily_return  GOOGL_daily_return  \
Date                                                                   
2020-01-02                NaN                NaN                 NaN   
2020-01-03          -0.002150          -0.004879            0.002341   
2020-01-06           0.004153          -0.001231            0.022839   
2020-01-07          -0.001891          -0.006306            0.000880   
2020-01-08           0.010757           0.010599            0.001788   

            JPM_daily_return  XOM_daily_return  
Date                                            
2020-01-02               NaN               NaN  
2020-01-03         -0.005624         -0.000467  
2020-01-06         -0.004610          0.003863  
2020-01-07         -0.014189         -0.005372  
2020-01-08          0.002471         -0.020410  


In [ ]:
import numpy as np

In [ ]:
def EWMA(window_data, wk: float): #做.apply()时由于切分下来的nparray会直接传入第一个参数，因此要把window_data写在最前面
    ewma=0
    period = len(window_data)
    for i in range(period):
        ewma += wk**(i-1)*window_data[period-i]
    return ewma
def ShortTermReversal(excess_return: pd.DataFrame(),ticker: str, halflife: int, period: int):
    if ticker not in excess_return.columns:
        return None
    wk = (0.5**(1/halflife))
    short_term_reversal=excess_return[ticker].rolling(period).apply(EWMA, raw=True, args=(wk,)) #rolling().apply()传递的是一个长为n的nparray，直接用[]索引
    return -short_term_reversal #反转因子信号是负的——过去收益率高，预期未来会回落

'''
def get_short_term_reversal_ewma(excess_return, ticker, period, halflife):
    if ticker not in excess_return.columns:
        return None
    
    price = excess_return[ticker]
    
    def ewma_func(window):
        weights = np.array([0.5**(1/halflife) ** (len(window) - 1 - i) for i in range(len(window))])
        weights /= weights.sum() #做权重归一化，不做归一化计算出来的EWMA值会受窗口期长短影响——窗口越长，权重之和越大，算出来的值越大，不同窗口期的值没有可比性
        return -np.dot(weights, window)
    
    return price.rolling(period).apply(ewma_func, raw=True)
    '''
    
    

- 历史波动率的计算公式通常是：Volatility= sqrt(Var(R))
- 在量化回测中，如果需要对满足某种特定条件的数据计算滚动指标，最聪明的办法是先将不满足条件的数据全部剔除（替换为 NaN），然后再执行滚动计算。

In [ ]:
def TwentyDayVolatility(daily_return, ticker): #获取20日波动率
    tdv=pd.DataFrame()
    if ticker not in daily_return.column:
        return None
    tdv[ticker]=daily_return[ticker].rolling(20).std()
    return tdv

def TwentyDayNegVotality(daily_return, ticker): #获取20日负收益波动率
    tdnv=pd.DataFrame()
    if ticker not in daily_return.column:
        return "ticker not in the data"
    s = daily_return[ticker]
    neg_return = s.where(s<0) #s.where(cond)的意思是：满足 cond 的保留，不满足的变成NaN
    tdnv[ticker] = neg_return.rolling(window=20, min_period = 1). std()
    return tdnv

In [24]:
volume = pd.DataFrame({t: data[t]["Volume"].squeeze() for t in tickers})

In [25]:
print(volume.head())

                 AAPL      MSFT     GOOGL       JPM       XOM       SPY
Date                                                                   
2020-01-02  135480400  22622100  27278000  10803700  12456400  59151200
2020-01-03  146322800  21116200  23408000  10386800  17386900  77709700
2020-01-06  118387200  20813700  46768000  10259000  20081900  55653900
2020-01-07  108872000  21634100  34330000  10531300  17387700  40496400
2020-01-08  132079200  27746500  35314000   9695300  15137700  68296000


- 20日平均成交量因子

In [ ]:
def TwentyDayAvgVol(volume, ticker):
    tdav=pd.DataFrame()
    tdav[ticker]=volume[ticker].rolling(20).mean()
    return tdav


- 将股票在短期内的量价走势分类为量价背离与量价同向， 并通过量价相关性来衡量量价走势的背离/同向程度。 通过回测， 我们发现量价相关性在半个月 的换仓周期下具有十分好的选股效果并具有显著的 Alpha。 在控制了 常见风险因子后， 该因子依旧对于股票收益具有较好的区分效果。

- 从一个较长的时间窗口上看，量与价的关系十分复杂，并有着多种多样的形态。这种复杂性使得我们很难使用量化指标把量与价结合起来进行分析。但从一个较短的时间窗口上去观察两者的走势时，量或者价的趋势性更强，这极大降低了分析的复杂性，也使得量价结合的指标构建变为可能。

- 中短期上，量价走势可以被简单分类为:“放量上涨”、“放量下跌”、“缩量上涨”、“缩量下跌”。而对于上述形态，可将其进一步简化为:“量价背离”与“量价同向”。而

- 对于量价的背离程度或者同向程度，可使用量价相关性进行衡量。量价相关性的具体定义为: 过去半个月中，股票复权收盘价与股票日换手率之间的 Pearson相关系数。

In [ ]:
def VolPriceCorr(volume, daily_return, ticker):
    rolling_corr=daily_returns[ticker].rolling(20).corr(volume[ticker])
    return rolling_corr